In [34]:
import pandas as pd 
import numpy as np 
import torch
from torchvision.datasets import MNIST

In [35]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

In [36]:
test_dataset = MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [37]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset,shuffle=True, batch_size=32, )
test_loader = DataLoader(test_dataset,shuffle=True,batch_size=32,)

In [38]:
for images, labels in train_loader:

    print(images.shape)
    print(labels.shape)

    break

torch.Size([32, 1, 28, 28])
torch.Size([32])


In [39]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        
        self.convo_layer = nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(64,128,kernel_size=3,),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        
        self.fc = nn.Sequential(
            
            nn.Flatten(),
            nn.Linear(128*1*1, 128),
            nn.ReLU(),

            nn.Linear(128,10)
          
        )

    def forward (self,x):
        x = self.convo_layer(x)
        output = self.fc(x)

        return output


In [40]:
import torch.optim as optim
model = CNN()
critarion = nn.CrossEntropyLoss()  
optim = optim.Adam(model.parameters())

In [42]:
epochs = 10
curr_loss = 0.0
epoch_loss = 0.0
best_epoch =float("inf") 

for epoch in range( epochs):
    
    optim.zero_grad()

    model.train()

    for images,label in train_loader:
        curr_loss = 0.0
        output = model(images)
        loss = critarion(output,label)
        loss.backward()
        optim.step()

        curr_loss += loss.item()
        
        epoch_loss+=curr_loss

    
    print(f"epoch : {epoch+1}/{epochs} current  total epoch_loss:{epoch_loss/len(train_loader)*100}")

model.eval()
test_loss=0.0

with torch.no_grad():
    for images, label in test_loader:
        output_test = model(images)
        loss_test = critarion(output_test,label)
        test_loss = loss_test.item()

        _,predicted = torch.max(output_test,1)
        correct = (predicted==label).sum()

    accuracy = 100 * correct / len(test_loader)

    print(f"Accuracy: {accuracy:.2f}%")
    


epoch : 1/10 current  total epoch_loss:231.65754544576006
epoch : 2/10 current  total epoch_loss:463.412515894572
epoch : 3/10 current  total epoch_loss:695.1070470809937
epoch : 4/10 current  total epoch_loss:926.4734528223673
epoch : 5/10 current  total epoch_loss:1157.5500819778442
epoch : 6/10 current  total epoch_loss:1388.5894584910075
epoch : 7/10 current  total epoch_loss:1619.7023156738283
epoch : 8/10 current  total epoch_loss:1850.9842904154461
epoch : 9/10 current  total epoch_loss:2082.129261868795
epoch : 10/10 current  total epoch_loss:2313.7919277572632
Accuracy: 0.96%


## RNN

In [43]:
import torch.nn as nn

class LSTMModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=28,
            hidden_size=128,
            batch_first=True
        )

        self.fc = nn.Linear(128, 10)

    def forward(self, x):

        # (batch,1,28,28) -> (batch,28,28)
        x = x.squeeze(1)

        output, (hidden, cell) = self.lstm(x)

        out = self.fc(hidden[-1])

        return out

In [45]:
import torch.optim as optim
model = LSTMModel()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# Training
epochs = 5

for epoch in range(epochs):

    model.train()

    epoch_loss = 0.0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss/len(train_loader):.4f}"
    )

#testing
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Epoch [1/5] Loss: 0.2956
Epoch [2/5] Loss: 0.0886
Epoch [3/5] Loss: 0.0629
Epoch [4/5] Loss: 0.0499
Epoch [5/5] Loss: 0.0423
Accuracy: 98.49%
